In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# 1. Create the input fields (Widgets)
name_input = widgets.Text(
    value='',
    placeholder='Type your name here...',
    description='Name:',
    disabled=False
)

age_slider = widgets.IntSlider(
    value=20,
    min=1,
    max=100,
    step=1,
    description='Age:',
    disabled=False,
    continuous_update=False # Updates only when you release the mouse
)

submit_button = widgets.Button(
    description='Submit App Data',
    button_style='success', # Gives it a green color
    icon='check'
)

# --- NEW: Create the Reset Button ---
reset_button = widgets.Button(
    description='Reset Form',
    button_style='danger', # Gives it a red color
    icon='undo'
)

# 2. Create an output area to display the results
output_area = widgets.Output()

# 3. Define what happens when you click the Submit button
def on_button_clicked(b):
    with output_area:
        clear_output() # Clears the previous message
        name = name_input.value
        age = age_slider.value
        
        if name.strip() == "":
            print("❌ Please enter a valid name first!")
        else:
            print(f"🎉 こんにちは {name}!")
            print(f"📊 あなたのプロフィールも {age}歳ですね。これからはあそぶ時間だよ！Lets go！")

# --- NEW: Define what happens when you click the Reset button ---
def on_reset_clicked(b):
    name_input.value = ''         # Wipes the text field
    age_slider.value = 20         # Resets slider to starting value
    with output_area:
        clear_output()            # Clears the printed greeting or error messages

# Connect the button click events to their respective functions
submit_button.on_click(on_button_clicked)
reset_button.on_click(on_reset_clicked)

# --- NEW: Group the buttons horizontally ---
button_box = widgets.HBox([submit_button, reset_button])

# 4. Display the app layout inside the cell
print("--- MY NATIVE JUPYTER APP ---")
display(name_input, age_slider, button_box, output_area)


--- MY NATIVE JUPYTER APP ---


Text(value='', description='Name:', placeholder='Type your name here...')

IntSlider(value=20, continuous_update=False, description='Age:', min=1)

Output()

In [9]:
import random
import time
import threading
import ipywidgets as widgets
from IPython.display import display, clear_output

# 1. Game Constants
EMOJIS = ['🎮', '🚀', '🤖', '👾', '🐱', '🍕', '🎨', '💎']
GAME_DURATION = 60  # 1 minute countdown limit

# Permanent Interface Canvas placeholders
timer_label = widgets.Label(value="⏰ Time Remaining: 60s")
grid_container = widgets.VBox()
status_output = widgets.Output()

# Dynamic Game State variables
GAME_ITEMS = []
revealed = []
first_choice = None
second_choice = None
matches_found = 0
buttons = []
time_left = GAME_DURATION
game_active = False
game_started = False
timer_thread = None

# 2. Async Background Clock Thread
def countdown_timer():
    global time_left, game_active
    while time_left > 0 and game_active:
        time.sleep(1)
        if not game_active:
            break
        time_left -= 1
        timer_label.value = f"⏰ Time Remaining: {time_left}s"
    
    # CASE: Timer ran out before finishing (FAIL / GAME OVER)
    if time_left <= 0 and game_active:
        game_active = False
        
        # Reveal all hidden emojis and turn them red
        for i in range(len(GAME_ITEMS)):
            buttons[i].disabled = True
            buttons[i].description = GAME_ITEMS[i]  
            if not revealed[i]:
                buttons[i].style.button_color = '#FFCDD2'  # Soft red for unmatched cards
            else:
                buttons[i].style.button_color = '#E0E0E0'  
                
        with status_output:
            clear_output()
            print("💀 GAME OVER! Time has run out! Click 'Restart Layout' to try again. 💀")

# 3. Board Reset & Initialization Controller
def setup_game_board():
    global GAME_ITEMS, revealed, first_choice, second_choice, matches_found, buttons, time_left, game_active, game_started
    
    game_active = False 
    game_started = False
    
    GAME_ITEMS = EMOJIS * 2
    random.shuffle(GAME_ITEMS)
    revealed = [False] * len(GAME_ITEMS)
    first_choice = None
    second_choice = None
    matches_found = 0
    time_left = GAME_DURATION
    
    timer_label.value = f"⏰ Time Remaining: {time_left}s"
    start_button.disabled = False  
    
    with status_output:
        clear_output()
        print("▶️ Click 'Start Game' below to begin the challenge!")
        
    buttons = []
    for i in range(len(GAME_ITEMS)):
        btn = widgets.Button(
            description='❓',
            layout=widgets.Layout(width='60px', height='60px', margin='4px'),
            disabled=True  # Locked tight until Start button is clicked
        )
        btn.style.button_color = '#757575' 
        
        btn.on_click(lambda b, idx=i: [on_any_click(None, idx, buttons), on_grid_click(idx, buttons)])
        buttons.append(btn)
        
    rows = [widgets.HBox(buttons[i:i+4]) for i in range(0, len(buttons), 4)]
    grid_container.children = rows

# 4. Input Trigger Events
def press_start_button(b):
    global game_active, game_started, timer_thread
    if game_started:
        return
    
    game_started = True
    game_active = True
    start_button.disabled = True  
    
    for btn in buttons:
        btn.disabled = False
        btn.style.button_color = '#2196F3' 
        
    with status_output:
        clear_output()
        print("🔥 Go! Find the matching pairs!")
        
    timer_thread = threading.Thread(target=countdown_timer, daemon=True)
    timer_thread.start()

def on_grid_click(button_index, buttons_list):
    global first_choice, second_choice, matches_found, game_active
    
    if not game_active or revealed[button_index] or (first_choice is not None and second_choice is not None):
        return
    
    buttons_list[button_index].description = GAME_ITEMS[button_index]
    buttons_list[button_index].style.button_color = '#E0E0E0'
    
    if first_choice is None:
        first_choice = button_index
    else:
        second_choice = button_index
        
        if GAME_ITEMS[first_choice] == GAME_ITEMS[second_choice]:
            revealed[first_choice] = True
            revealed[second_choice] = True
            matches_found += 1
            
            with status_output:
                clear_output()
                # CASE: Managed to solve everything within the timeframe (SUCCESS)
                if matches_found == len(EMOJIS):
                    game_active = False # Stops the timer thread instantly
                    
                    # Turn all cards gold to celebrate victory!
                    for btn in buttons_list:
                        btn.style.button_color = '#FFD700' 
                        btn.disabled = True
                        
                    print("✨ やった! あなたのスコアはすごい! ✨")
                    print(f"⏱️ Finished with {time_left} seconds left on the clock.")
                else:
                    print(f"✅ Match found! Total Progress: {matches_found}/{len(EMOJIS)}")
                    
            first_choice = None
            second_choice = None
        else:
            with status_output:
                clear_output()
                print("❌ No match! Click another card to reset them...")

def on_any_click(change, index, buttons):
    global first_choice, second_choice
    if first_choice is not None and second_choice is not None:
        buttons[first_choice].description = '❓'
        buttons[first_choice].style.button_color = '#2196F3'
        buttons[second_choice].description = '❓'
        buttons[second_choice].style.button_color = '#2196F3'
        first_choice = None
        second_choice = None

# 5. Build Control Layout Widgets
start_button = widgets.Button(
    description='Start Game',
    button_style='primary', 
    icon='play',
    layout=widgets.Layout(margin='10px 4px 0px 4px')
)
start_button.on_click(press_start_button)

restart_button = widgets.Button(
    description='Restart Layout',
    button_style='success', 
    icon='undo',
    layout=widgets.Layout(margin='10px 4px 0px 4px')
)
restart_button.on_click(lambda b: setup_game_board())

control_box = widgets.HBox([start_button, restart_button])

# 6. Render Window
print("🕹️ JUPYTER MEMORY MATCH GAME 🕹️")
setup_game_board()

display(timer_label)
display(grid_container)
display(control_box)
display(status_output)


🕹️ JUPYTER MEMORY MATCH GAME 🕹️


Label(value='⏰ Time Remaining: 60s')

Output()